# ViT + Metadata Fusion Model Analysis

This notebook provides comprehensive analysis of the trained MushroomViTWithMetadata model:
- Learning curves
- Confusion matrix for hardest classes
- Worst performing class examples
- Attention map visualization
- Per-class accuracy breakdown
- CNN vs ViT comparison

In [ ]:
import torch
import timm
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from PIL import Image
from torch.utils.data import DataLoader
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import torch.nn.functional as F

# Style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 10, 'axes.titlesize': 14, 'axes.labelsize': 12})

# Create outputs directory
os.makedirs('outputs', exist_ok=True)

# Paths - adjust for your environment
CHECKPOINT_PATH = '/content/drive/MyDrive/mushroom_checkpoints/best_model.pth'
DATA_CSV = 'data/raw/DF20M-metadata/DF20M-train_metadata_PROD.csv'
DATA_DIR = 'data/raw/DF20M'

# Add project root to path
sys.path.insert(0, '/content/Applied_Mashroomatics')

# Load checkpoint
print("Loading checkpoint...")
ckpt = torch.load(CHECKPOINT_PATH, map_location='cpu')

habitat_vocab = ckpt['habitat_vocab']
substrate_vocab = ckpt['substrate_vocab']
NUM_CLASSES = ckpt['num_classes']
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"Device: {DEVICE}")
print(f"Classes: {NUM_CLASSES}")
print(f"Val accuracy from training: {ckpt.get('val_acc', 'N/A')}%")

# Load data and create species mapping
df = pd.read_csv(DATA_CSV)
df = df.dropna(subset=['species']).reset_index(drop=True)
species_list = sorted(df['species'].unique())
species_to_id = {sp: i for i, sp in enumerate(species_list)}
id_to_species = {i: sp for sp, i in species_to_id.items()}

# Split data
unique_ids = df['gbifID'].unique()
_, val_ids = train_test_split(unique_ids, test_size=0.2, random_state=42)
val_df = df[df['gbifID'].isin(val_ids)].reset_index(drop=True)
print(f"Val samples: {len(val_df)}")

# Load model
from src.models.mushroom_vit import MushroomViTWithMetadata
from src.data.transforms import get_transforms

model = MushroomViTWithMetadata(
    num_classes=NUM_CLASSES,
    habitat_vocab=habitat_vocab,
    substrate_vocab=substrate_vocab,
    pretrained=False
)
model.load_state_dict(ckpt['model_state_dict'])
model = model.to(DEVICE).eval()
print("Model loaded!")

## Section 1: Learning Curves

Visualize training progress to understand:
- How quickly the model converged
- Whether overfitting occurred (train-val gap)
- When the best model was saved

In [ ]:
# Training logs - paste your epoch data here or load from checkpoint
# Format: list of dicts with keys: epoch, train_loss, val_loss, train_acc, val_acc

training_logs = [
    # Paste your training logs here, e.g.:
    # {'epoch': 1, 'train_loss': 5.18, 'val_loss': 4.99, 'train_acc': 1.47, 'val_acc': 2.99},
    # {'epoch': 2, 'train_loss': 4.84, 'val_loss': 4.62, 'train_acc': 6.22, 'val_acc': 10.22},
    # ...
]

if len(training_logs) == 0:
    print("No training logs provided. Please paste your epoch data above.")
    print("Example format:")
    print("training_logs = [")
    print("    {'epoch': 1, 'train_loss': 5.18, 'val_loss': 4.99, 'train_acc': 1.47, 'val_acc': 2.99},")
    print("    ...")
    print("]")
else:
    logs_df = pd.DataFrame(training_logs)
    best_epoch = logs_df.loc[logs_df['val_acc'].idxmax(), 'epoch']
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Loss plot
    axes[0].plot(logs_df['epoch'], logs_df['train_loss'], color='#1D9E75', 
                 label='Train Loss', linewidth=2, marker='o', markersize=4)
    axes[0].plot(logs_df['epoch'], logs_df['val_loss'], color='#D85A30',
                 label='Val Loss', linewidth=2, marker='s', markersize=4)
    axes[0].axvline(x=best_epoch, color='gray', linestyle='--', alpha=0.7, label=f'Best epoch ({best_epoch})')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Training and Validation Loss')
    axes[0].legend()
    
    # Accuracy plot
    axes[1].plot(logs_df['epoch'], logs_df['train_acc'], color='#1D9E75',
                 label='Train Acc', linewidth=2, marker='o', markersize=4)
    axes[1].plot(logs_df['epoch'], logs_df['val_acc'], color='#D85A30',
                 label='Val Acc', linewidth=2, marker='s', markersize=4)
    axes[1].axvline(x=best_epoch, color='gray', linestyle='--', alpha=0.7, label=f'Best epoch ({best_epoch})')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy (%)')
    axes[1].set_title('Training and Validation Accuracy')
    axes[1].legend()
    
    plt.tight_layout()
    plt.savefig('outputs/01_learning_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"\nBest validation accuracy: {logs_df['val_acc'].max():.2f}% at epoch {best_epoch}")

## Section 2: Confusion Matrix (Top-20 Hardest Classes)

Identify which species are most commonly confused with each other.
This helps understand:
- Visually similar species that the model struggles with
- Potential label noise in the dataset
- Areas where metadata fusion could help more

In [ ]:
# Run inference and save predictions (or load if already exists)
PREDICTIONS_FILE = 'outputs/predictions.npz'

if os.path.exists(PREDICTIONS_FILE):
    print("Loading cached predictions...")
    data = np.load(PREDICTIONS_FILE)
    all_preds = data['preds']
    all_labels = data['labels']
    all_probs = data['probs'] if 'probs' in data else None
else:
    print("Running inference on validation set...")
    
    val_transform = get_transforms('val')
    all_preds = []
    all_labels = []
    all_probs = []
    
    h_col = 'Habitat' if 'Habitat' in val_df.columns else 'habitat'
    s_col = 'Substrate' if 'Substrate' in val_df.columns else 'substrate'
    
    with torch.no_grad():
        for idx in tqdm(range(len(val_df)), desc="Inference"):
            row = val_df.iloc[idx]
            img_path = os.path.join(DATA_DIR, row['image_path'])
            
            try:
                image = Image.open(img_path).convert('RGB')
                image = val_transform(image).unsqueeze(0).to(DEVICE)
            except:
                continue
            
            label = species_to_id[row['species']]
            
            h_val = row.get(h_col)
            s_val = row.get(s_col)
            h_id = habitat_vocab.get(h_val, len(habitat_vocab)) if pd.notna(h_val) else len(habitat_vocab)
            s_id = substrate_vocab.get(s_val, len(substrate_vocab)) if pd.notna(s_val) else len(substrate_vocab)
            month = int(row['month']) if pd.notna(row.get('month')) else 6
            
            h = torch.tensor([h_id]).to(DEVICE)
            s = torch.tensor([s_id]).to(DEVICE)
            m = torch.tensor([month]).to(DEVICE)
            
            output = model(image, h, s, m)
            probs = F.softmax(output, dim=1)
            pred = output.argmax(1).item()
            
            all_preds.append(pred)
            all_labels.append(label)
            all_probs.append(probs.cpu().numpy()[0])
    
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)
    
    np.savez(PREDICTIONS_FILE, preds=all_preds, labels=all_labels, probs=all_probs)
    print(f"Predictions saved to {PREDICTIONS_FILE}")

accuracy = (all_preds == all_labels).mean() * 100
print(f"\nOverall accuracy: {accuracy:.2f}%")
print(f"Total samples: {len(all_labels)}")

In [ ]:
# Find top-20 classes with most errors
errors_per_class = {}
for i in range(NUM_CLASSES):
    mask = all_labels == i
    if mask.sum() > 0:
        errors = (all_preds[mask] != all_labels[mask]).sum()
        errors_per_class[i] = errors

worst_classes = sorted(errors_per_class.keys(), key=lambda x: errors_per_class[x], reverse=True)[:20]
worst_class_names = [id_to_species[c][:15] for c in worst_classes]

print("Top-20 classes with most errors:")
for c in worst_classes[:10]:
    print(f"  {id_to_species[c]}: {errors_per_class[c]} errors")

# Filter predictions for worst classes
mask = np.isin(all_labels, worst_classes)
filtered_labels = all_labels[mask]
filtered_preds = all_preds[mask]

# Remap to 0-19 for confusion matrix
class_map = {c: i for i, c in enumerate(worst_classes)}
mapped_labels = np.array([class_map.get(l, -1) for l in filtered_labels])
mapped_preds = np.array([class_map.get(p, -1) for p in filtered_preds])

# Filter out predictions that aren't in top-20
valid_mask = (mapped_labels >= 0) & (mapped_preds >= 0)
mapped_labels = mapped_labels[valid_mask]
mapped_preds = mapped_preds[valid_mask]

# Confusion matrix
cm = confusion_matrix(mapped_labels, mapped_preds, labels=range(20))

plt.figure(figsize=(16, 14))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=worst_class_names, yticklabels=worst_class_names,
            mask=(cm == 0), cbar_kws={'label': 'Count'})
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix: Top-20 Most Misclassified Species')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('outputs/02_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Classification report
report = classification_report(all_labels, all_preds, 
                               target_names=[id_to_species[i] for i in range(NUM_CLASSES)],
                               zero_division=0)
print(report)

with open('outputs/02_classification_report.txt', 'w') as f:
    f.write(report)
print("\nClassification report saved to outputs/02_classification_report.txt")

## Section 3: Worst Class Examples

Visual inspection of the hardest species helps understand:
- Image quality issues (blur, occlusion, lighting)
- Intra-class variation
- Which features the model might be missing

In [ ]:
# Load predictions if needed
if 'all_preds' not in dir():
    data = np.load('outputs/predictions.npz')
    all_preds = data['preds']
    all_labels = data['labels']

# Calculate per-class accuracy
class_accuracy = {}
for i in range(NUM_CLASSES):
    mask = all_labels == i
    if mask.sum() > 0:
        acc = (all_preds[mask] == all_labels[mask]).mean() * 100
        class_accuracy[i] = acc

# Top-8 worst classes
worst_8 = sorted(class_accuracy.keys(), key=lambda x: class_accuracy[x])[:8]

print("Top-8 worst performing classes:")
for c in worst_8:
    print(f"  {id_to_species[c]}: {class_accuracy[c]:.1f}%")

In [ ]:
# Find examples for worst classes
val_transform = get_transforms('val')
h_col = 'Habitat' if 'Habitat' in val_df.columns else 'habitat'
s_col = 'Substrate' if 'Substrate' in val_df.columns else 'substrate'

fig, axes = plt.subplots(8, 4, figsize=(16, 32))

for row_idx, class_id in enumerate(worst_8):
    class_name = id_to_species[class_id]
    mask = all_labels == class_id
    indices = np.where(mask)[0]
    
    correct_idx = indices[all_preds[indices] == class_id][:2]
    wrong_idx = indices[all_preds[indices] != class_id][:2]
    
    examples = list(correct_idx) + list(wrong_idx)
    
    for col_idx in range(4):
        ax = axes[row_idx, col_idx]
        
        if col_idx < len(examples):
            idx = examples[col_idx]
            row = val_df.iloc[idx]
            img_path = os.path.join(DATA_DIR, row['image_path'])
            
            try:
                img = Image.open(img_path).convert('RGB')
                ax.imshow(img)
                
                is_correct = all_preds[idx] == all_labels[idx]
                if is_correct:
                    ax.set_title(f"Correct", fontsize=10, color='green')
                else:
                    pred_name = id_to_species[all_preds[idx]][:20]
                    ax.set_title(f"Pred: {pred_name}", fontsize=9, color='red')
            except:
                ax.text(0.5, 0.5, 'Image not found', ha='center', va='center')
        
        ax.axis('off')
        
        if col_idx == 0:
            short_name = class_name[:25] + '...' if len(class_name) > 25 else class_name
            ax.set_ylabel(f"{short_name}\n({class_accuracy[class_id]:.0f}%)", 
                         fontsize=10, rotation=0, ha='right', va='center')

plt.suptitle('Worst Performing Classes: Correct vs Wrong Predictions', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('outputs/03_worst_classes.png', dpi=150, bbox_inches='tight')
plt.show()

## Section 4: Attention Maps

Visualize where the model "looks" when making predictions.
This reveals:
- Whether the model focuses on the mushroom vs background
- If specific morphological features are being used
- Potential issues with attention (e.g., focusing on irrelevant areas)

In [ ]:
def get_attention_map(model, image_tensor, h_id, s_id, month):
    """Extract attention map from last transformer block."""
    attention_weights = None
    
    def hook_fn(module, input, output):
        nonlocal attention_weights
        # output is tuple (attn_output, attn_weights) for some implementations
        # For timm ViT, we need to access the attention differently
        pass
    
    # Alternative: compute attention manually
    model.eval()
    with torch.no_grad():
        B = image_tensor.shape[0]
        
        # Patch embedding
        x = model.vit.patch_embed(image_tensor)
        
        # Add CLS token and positional embedding
        cls_token = model.vit.cls_token.expand(B, -1, -1)
        x = torch.cat([cls_token, x], dim=1)
        x = x + model.vit.pos_embed
        
        # Add metadata token
        meta_feat = model.meta_encoder(h_id, s_id, month).unsqueeze(1)
        meta_feat = meta_feat + model.meta_token
        x = torch.cat([x[:, :1], meta_feat, x[:, 1:]], dim=1)
        
        x = model.vit.pos_drop(x)
        
        # Run through all blocks except last
        for block in model.vit.blocks[:-1]:
            x = block(x)
        
        # Last block - extract attention
        last_block = model.vit.blocks[-1]
        
        # Manually compute attention in last block
        y = last_block.norm1(x)
        B, N, C = y.shape
        
        qkv = last_block.attn.qkv(y).reshape(B, N, 3, last_block.attn.num_heads, C // last_block.attn.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        
        attn = (q @ k.transpose(-2, -1)) * last_block.attn.scale
        attn = attn.softmax(dim=-1)
        
        # CLS token attention to patch tokens (skip CLS and meta tokens)
        # Shape: (B, heads, tokens, tokens)
        # We want: attn[0, :, 0, 2:] - CLS attending to patches (skip CLS at 0, meta at 1)
        cls_attn = attn[0, :, 0, 2:]  # (heads, 196)
        cls_attn = cls_attn.mean(dim=0)  # Average across heads
        
        # Reshape to 14x14 and upsample
        cls_attn = cls_attn.reshape(14, 14)
        cls_attn = F.interpolate(cls_attn.unsqueeze(0).unsqueeze(0), 
                                  size=(224, 224), mode='bilinear', align_corners=False)
        cls_attn = cls_attn.squeeze().cpu().numpy()
        
        # Normalize
        cls_attn = (cls_attn - cls_attn.min()) / (cls_attn.max() - cls_attn.min() + 1e-8)
        
    return cls_attn

print("Attention map function defined.")

In [ ]:
# Select 12 images: 4 correct high-conf, 4 correct low-conf, 4 wrong
if 'all_probs' not in dir() or all_probs is None:
    data = np.load('outputs/predictions.npz')
    all_probs = data['probs']

confidences = all_probs.max(axis=1)
correct_mask = all_preds == all_labels
wrong_mask = ~correct_mask

# High confidence correct
correct_indices = np.where(correct_mask)[0]
correct_conf = confidences[correct_indices]
high_conf_correct = correct_indices[np.argsort(correct_conf)[-4:]]

# Low confidence correct
low_conf_correct = correct_indices[np.argsort(correct_conf)[:4]]

# Wrong predictions
wrong_indices = np.where(wrong_mask)[0]
wrong_examples = wrong_indices[:4]

selected_indices = list(high_conf_correct) + list(low_conf_correct) + list(wrong_examples)
categories = ['High-conf correct']*4 + ['Low-conf correct']*4 + ['Wrong prediction']*4

print(f"Selected {len(selected_indices)} images for attention visualization")

In [ ]:
val_transform = get_transforms('val')
h_col = 'Habitat' if 'Habitat' in val_df.columns else 'habitat'
s_col = 'Substrate' if 'Substrate' in val_df.columns else 'substrate'

fig, axes = plt.subplots(12, 2, figsize=(10, 48))

for i, (idx, cat) in enumerate(zip(selected_indices, categories)):
    row = val_df.iloc[idx]
    img_path = os.path.join(DATA_DIR, row['image_path'])
    
    try:
        img_pil = Image.open(img_path).convert('RGB')
        img_tensor = val_transform(img_pil).unsqueeze(0).to(DEVICE)
        
        h_val = row.get(h_col)
        s_val = row.get(s_col)
        h_id = torch.tensor([habitat_vocab.get(h_val, len(habitat_vocab)) if pd.notna(h_val) else len(habitat_vocab)]).to(DEVICE)
        s_id = torch.tensor([substrate_vocab.get(s_val, len(substrate_vocab)) if pd.notna(s_val) else len(substrate_vocab)]).to(DEVICE)
        month = torch.tensor([int(row['month']) if pd.notna(row.get('month')) else 6]).to(DEVICE)
        
        attn_map = get_attention_map(model, img_tensor, h_id, s_id, month)
        
        # Original image
        img_resized = img_pil.resize((224, 224))
        axes[i, 0].imshow(img_resized)
        axes[i, 0].set_title(f"{cat}\n{id_to_species[all_labels[idx]][:30]}\nConf: {confidences[idx]:.2f}", fontsize=9)
        axes[i, 0].axis('off')
        
        # Attention overlay
        axes[i, 1].imshow(img_resized)
        axes[i, 1].imshow(attn_map, cmap='hot', alpha=0.5)
        if all_preds[idx] != all_labels[idx]:
            axes[i, 1].set_title(f"Pred: {id_to_species[all_preds[idx]][:30]}", fontsize=9, color='red')
        else:
            axes[i, 1].set_title("Attention Map", fontsize=9)
        axes[i, 1].axis('off')
        
    except Exception as e:
        print(f"Error processing image {idx}: {e}")
        axes[i, 0].text(0.5, 0.5, 'Error', ha='center', va='center')
        axes[i, 1].text(0.5, 0.5, 'Error', ha='center', va='center')

plt.suptitle('Attention Maps: Where the Model Looks', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('outputs/04_attention_maps.png', dpi=150, bbox_inches='tight')
plt.show()

## Section 5: Per-Class Accuracy Bar Chart

Overview of model performance across all species.
Identifies:
- Which species are "easy" vs "hard"
- Class imbalance effects
- Potential data quality issues for specific species

In [ ]:
# Load predictions if needed
if 'all_preds' not in dir():
    data = np.load('outputs/predictions.npz')
    all_preds = data['preds']
    all_labels = data['labels']

# Calculate per-class stats
class_stats = []
for i in range(NUM_CLASSES):
    mask = all_labels == i
    count = mask.sum()
    if count > 0:
        acc = (all_preds[mask] == all_labels[mask]).mean() * 100
    else:
        acc = 0
    class_stats.append({
        'class_id': i,
        'species': id_to_species[i],
        'accuracy': acc,
        'count': count
    })

stats_df = pd.DataFrame(class_stats)
stats_df = stats_df.sort_values('accuracy', ascending=True).reset_index(drop=True)

mean_acc = stats_df['accuracy'].mean()

print(f"Mean per-class accuracy: {mean_acc:.2f}%")
print(f"Classes with >80% accuracy: {(stats_df['accuracy'] >= 80).sum()} ({(stats_df['accuracy'] >= 80).mean()*100:.1f}%)")
print(f"Classes with 50-80% accuracy: {((stats_df['accuracy'] >= 50) & (stats_df['accuracy'] < 80)).sum()}")
print(f"Classes with <50% accuracy: {(stats_df['accuracy'] < 50).sum()} ({(stats_df['accuracy'] < 50).mean()*100:.1f}%)")

In [ ]:
# Bar chart
fig, ax = plt.subplots(figsize=(12, 40))

colors = []
for acc in stats_df['accuracy']:
    if acc >= 80:
        colors.append('#2ECC71')  # Green
    elif acc >= 50:
        colors.append('#F39C12')  # Amber
    else:
        colors.append('#E74C3C')  # Red

y_pos = range(len(stats_df))
bars = ax.barh(y_pos, stats_df['accuracy'], color=colors, height=0.8)

# Add count labels
for i, (acc, count) in enumerate(zip(stats_df['accuracy'], stats_df['count'])):
    ax.text(acc + 1, i, f'n={count}', va='center', fontsize=6)

ax.axvline(x=mean_acc, color='black', linestyle='--', linewidth=1.5, label=f'Mean: {mean_acc:.1f}%')
ax.set_yticks(y_pos)
ax.set_yticklabels([s[:30] for s in stats_df['species']], fontsize=6)
ax.set_xlabel('Accuracy (%)')
ax.set_title('Per-Class Accuracy (sorted ascending)')
ax.legend(loc='lower right')
ax.set_xlim(0, 105)

plt.tight_layout()
plt.savefig('outputs/05_per_class_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Top 10 easiest and hardest
print("\nTop-10 EASIEST classes (highest accuracy):")
for _, row in stats_df.tail(10).iloc[::-1].iterrows():
    print(f"  {row['species'][:40]}: {row['accuracy']:.1f}% (n={row['count']})")

print("\nTop-10 HARDEST classes (lowest accuracy):")
for _, row in stats_df.head(10).iterrows():
    print(f"  {row['species'][:40]}: {row['accuracy']:.1f}% (n={row['count']})")

## Section 6: CNN vs ViT Error Comparison

Compare where ViT succeeds but CNN fails, and vice versa.
This reveals:
- Complementary strengths of different architectures
- Potential for ensemble methods
- Cases that are fundamentally hard for all models

In [ ]:
CNN_PREDICTIONS_FILE = 'outputs/cnn_predictions.npz'

if not os.path.exists(CNN_PREDICTIONS_FILE):
    print("CNN predictions not found, skipping section 6.")
    print("\nTo generate CNN predictions, run:")
    print("")
    print("  1. Load your trained CNN model")
    print("  2. Run inference on the same validation set")
    print("  3. Save predictions:")
    print("     np.savez('outputs/cnn_predictions.npz', preds=cnn_preds, labels=cnn_labels)")
    print("")
    print("Then re-run this cell.")
else:
    # Load CNN predictions
    cnn_data = np.load(CNN_PREDICTIONS_FILE)
    cnn_preds = cnn_data['preds']
    cnn_labels = cnn_data['labels']
    
    # Load ViT predictions
    vit_data = np.load('outputs/predictions.npz')
    vit_preds = vit_data['preds']
    vit_labels = vit_data['labels']
    
    # Ensure same size
    min_len = min(len(cnn_preds), len(vit_preds))
    cnn_preds = cnn_preds[:min_len]
    vit_preds = vit_preds[:min_len]
    labels = vit_labels[:min_len]
    
    # Compare
    vit_correct = vit_preds == labels
    cnn_correct = cnn_preds == labels
    
    vit_wins = vit_correct & ~cnn_correct  # ViT correct, CNN wrong
    cnn_wins = cnn_correct & ~vit_correct  # CNN correct, ViT wrong
    both_correct = vit_correct & cnn_correct
    both_wrong = ~vit_correct & ~cnn_correct
    
    print("="*50)
    print("CNN vs ViT Comparison")
    print("="*50)
    print(f"ViT correct, CNN wrong: {vit_wins.sum()} ({vit_wins.mean()*100:.1f}%)")
    print(f"CNN correct, ViT wrong: {cnn_wins.sum()} ({cnn_wins.mean()*100:.1f}%)")
    print(f"Both correct:           {both_correct.sum()} ({both_correct.mean()*100:.1f}%)")
    print(f"Both wrong:             {both_wrong.sum()} ({both_wrong.mean()*100:.1f}%)")
    print("="*50)

In [ ]:
if os.path.exists(CNN_PREDICTIONS_FILE):
    # Visual comparison grid
    fig, axes = plt.subplots(3, 6, figsize=(18, 9))
    
    categories_data = [
        ('ViT wins', np.where(vit_wins)[0]),
        ('CNN wins', np.where(cnn_wins)[0]),
        ('Both wrong', np.where(both_wrong)[0])
    ]
    
    for row_idx, (cat_name, indices) in enumerate(categories_data):
        examples = indices[:6] if len(indices) >= 6 else indices
        
        for col_idx in range(6):
            ax = axes[row_idx, col_idx]
            
            if col_idx < len(examples):
                idx = examples[col_idx]
                row = val_df.iloc[idx]
                img_path = os.path.join(DATA_DIR, row['image_path'])
                
                try:
                    img = Image.open(img_path).convert('RGB')
                    ax.imshow(img)
                    
                    true_label = id_to_species[labels[idx]][:15]
                    vit_pred = id_to_species[vit_preds[idx]][:15]
                    cnn_pred = id_to_species[cnn_preds[idx]][:15]
                    
                    ax.set_title(f"True: {true_label}\nViT: {vit_pred}\nCNN: {cnn_pred}", 
                                fontsize=7)
                except:
                    ax.text(0.5, 0.5, 'Error', ha='center', va='center')
            
            ax.axis('off')
            
            if col_idx == 0:
                ax.set_ylabel(cat_name, fontsize=12, rotation=0, ha='right', va='center')
    
    plt.suptitle('CNN vs ViT: Error Comparison', fontsize=14)
    plt.tight_layout()
    plt.savefig('outputs/06_cnn_vs_vit.png', dpi=150, bbox_inches='tight')
    plt.show()

## Summary

### Key Findings

- **Best validation accuracy:** [fill from training]
- **Hardest species:** [fill from Section 5]
- **Easiest species:** [fill from Section 5]
- **Attention maps show the model focuses on:** [fill manually after viewing Section 4]

### Recommendations

1. [Add based on analysis]
2. [Add based on analysis]
3. [Add based on analysis]